In [1]:
import pickle
import networkx as nx
import polars as pl

In [2]:
with open('../../data/raw/eth.pkl', 'rb') as ethfile:
    eth = pickle.load(ethfile)

In [3]:
nx.__version__

'3.6.1'

In [4]:
eth.number_of_nodes()

2973489

In [5]:
eth.number_of_edges()

13551303

In [ ]:
edge_data = [
    (u, v, d['amount'], d['timestamp'])
    for u, v, k, d in eth.edges(keys=True, data=True)
]

df_edges = pl.DataFrame(edge_data, schema=['source', 'target', 'amount', 'timestamp'])

In [7]:
df_edges.head()

source,target,amount,timestamp
str,str,f64,f64
"""0x1f1e784a61a8ca0a90250bcd2170…","""0x1266f8b9e4dffc9e2f719bf51713…",2.3446233,1.5265e9
"""0x1f1e784a61a8ca0a90250bcd2170…","""0x806ceb189d36700a97f4e7ecd4fb…",0.07,1.5045e9
"""0x1f1e784a61a8ca0a90250bcd2170…","""0x806ceb189d36700a97f4e7ecd4fb…",0.052111,1.5045e9
"""0x1f1e784a61a8ca0a90250bcd2170…","""0x3ec4688db6bf8464b0bef30ec2ca…",5.068543,1.5078e9
"""0x1f1e784a61a8ca0a90250bcd2170…","""0x3ec4688db6bf8464b0bef30ec2ca…",0.9925,1.5271e9


In [9]:
df_nodes = pl.DataFrame([
    (n, d.get('isp', None)) for n, d in eth.nodes(data=True)
], schema=['node_id', 'isp'])

/var/folders/0x/klqdyhpj427dvg81kxzyfcvm0000gn/T/ipykernel_27784/1382913157.py:1: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df_nodes = pl.DataFrame([


In [11]:
df_nodes.head()

node_id,isp
str,i64
"""0x1f1e784a61a8ca0a90250bcd2170…",0
"""0x1266f8b9e4dffc9e2f719bf51713…",0
"""0xbbfaf27674c2eb5d13edc58a4008…",1
"""0x256fc19e9d8f5be0d451841f2182…",0
"""0xb50d0c4cb2c29cc232c96a59e9c6…",0


In [21]:
timestamps = df_edges.select(pl.col('timestamp')).unique().with_columns(pl.from_epoch('timestamp', time_unit='s'))
timestamps.head()

timestamp
datetime[μs]
2018-03-24 23:42:16
2017-12-21 00:22:50
2017-12-22 07:21:18
2017-09-01 23:20:34
2018-08-04 09:39:25


In [22]:
zero = timestamps.min()
zero

timestamp
datetime[μs]
2015-08-07 05:01:09


In [23]:
hundred = timestamps.max()
hundred

timestamp
datetime[μs]
2019-01-19 09:24:22


In [24]:
diff = hundred - zero
diff

timestamp
duration[μs]
1261d 4h 23m 13s
